# EDA Media Crypto (Bronze, Silver, Gold)

Objectif:
- Faire un diagnostic rapide des datasets `bronze`, `silver`, `gold`
- Mettre l'accent sur l'analyse du `gold` (series temporelles et sources media)
- Produire des indicateurs exploitables pour la suite ML

Ce notebook est concu pour etre versionne tel quel dans Git (structure prechargee).

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
BRONZE_PATH = DATA_DIR / "bronze" / "bitcoin_bronze.jsonl"
SILVER_PATH = DATA_DIR / "silver" / "bitcoin_silver.csv"
GOLD_PATH = DATA_DIR / "gold" / "bitcoin_gold.csv"

print("BASE_DIR:", BASE_DIR)
print("BRONZE:", BRONZE_PATH)
print("SILVER:", SILVER_PATH)
print("GOLD:", GOLD_PATH)

BASE_DIR: /home/herrahj/test_crypto/crypto-prediction
BRONZE: /home/herrahj/test_crypto/crypto-prediction/data/bronze/bitcoin_bronze.jsonl
SILVER: /home/herrahj/test_crypto/crypto-prediction/data/silver/bitcoin_silver.csv
GOLD: /home/herrahj/test_crypto/crypto-prediction/data/gold/bitcoin_gold.csv


In [2]:
# Chargement des trois niveaux de donnees
bronze_df = pd.read_json(BRONZE_PATH, lines=True)
silver_df = pd.read_csv(SILVER_PATH)
gold_df = pd.read_csv(GOLD_PATH)

# Dates pour faciliter les analyses temporelles
silver_df["date"] = pd.to_datetime(silver_df["date"], errors="coerce")
gold_df["date"] = pd.to_datetime(gold_df["date"], errors="coerce")

print("Shapes:")
print("- bronze:", bronze_df.shape)
print("- silver:", silver_df.shape)
print("- gold  :", gold_df.shape)

print("\nColonnes:")
print("- bronze:", list(bronze_df.columns))
print("- silver:", list(silver_df.columns))
print("- gold  :", list(gold_df.columns))

print("\nPerimetre temporel:")
print("- silver:", silver_df["date"].min(), "->", silver_df["date"].max())
print("- gold  :", gold_df["date"].min(), "->", gold_df["date"].max())

print("\nApercu:")
display(bronze_df.head(2))
display(silver_df.head(2))
display(gold_df.head(2))

Shapes:
- bronze: (4023242, 3)
- silver: (4018370, 3)
- gold  : (4035, 18)

Colonnes:
- bronze: ['url', 'seendate', 'domain']
- silver: ['date', 'url', 'domain']
- gold  : ['date', 'BBC', 'Bitcoin.com', 'Bloomberg', 'CNN', 'CoinDesk', 'CoinSpeaker', 'Cointelegraph', 'Decrypt', 'Financial Times', 'France 24', 'Le Figaro', 'Le Monde', 'Les Échos', 'New York Times', 'Other', 'Reuters', 'total']

Perimetre temporel:
- silver: 2015-02-18 00:00:00 -> 2026-03-24 00:00:00
- gold  : 2015-02-18 00:00:00 -> 2026-03-24 00:00:00

Apercu:


,url,seendate,domain
0,https://news.bitcoin.com/over-200-venezuelan-t...,2019-10-10,news.bitcoin.com
1,https://www.edaily.co.kr/news/read?newsId=0218...,2019-10-10,edaily.co.kr


,date,url,domain
0,2019-10-10,https://news.bitcoin.com/over-200-venezuelan-t...,news.bitcoin.com
1,2019-10-10,https://www.edaily.co.kr/news/read?newsId=0218...,edaily.co.kr


,date,BBC,Bitcoin.com,Bloomberg,CNN,CoinDesk,CoinSpeaker,Cointelegraph,Decrypt,Financial Times,France 24,Le Figaro,Le Monde,Les Échos,New York Times,Other,Reuters,total
0,2015-02-18,0,0,0,1,0,0,0,0,0,0,0,0,0,0,38,0,39
1,2015-02-19,0,0,1,1,7,0,1,0,0,0,0,0,0,2,198,1,211


## Controle rapide Bronze / Silver

Objectif:
- Verifier la qualite et la coherence de la preparation avant l'agregation `gold`
- Mesurer rapidement la duplication, les valeurs manquantes et la couverture domaine

In [3]:
# Bronze: champs manquants
bronze_missing = bronze_df[["url", "seendate", "domain"]].isna().mean().sort_values(ascending=False)
print("Taux de valeurs manquantes - bronze")
display((bronze_missing * 100).round(2).to_frame("missing_pct"))

# Silver: duplications URL + dates nulles
silver_url_dupes = silver_df["url"].duplicated().sum()
silver_date_na = silver_df["date"].isna().sum()
print("Duplicats URL silver:", silver_url_dupes)
print("Dates invalides/nan silver:", silver_date_na)

# Domaines les plus frequents
print("\nTop 20 domaines silver")
display(silver_df["domain"].value_counts(dropna=False).head(20).to_frame("count"))

Taux de valeurs manquantes - bronze


,missing_pct
url,0.0
seendate,0.0
domain,0.0


Duplicats URL silver: 0
Dates invalides/nan silver: 0

Top 20 domaines silver


,count
domain,
dailypolitical.com,131464
wkrb13.com,127247
modernreaders.com,111508
biztoc.com,98489
themarketsdaily.com,76285
theenterpriseleader.com,58465
newsbtc.com,56580
tickerreport.com,40366
insidebitcoins.com,38303


## EDA Gold (focus principal)

Ici on analyse surtout:
- La dynamique temporelle du volume total
- La contribution des sources media
- Les pics/jours atypiques et la concentration du coverage

In [4]:
# Identification des colonnes sources
non_source_cols = {"date", "total"}
source_cols = [c for c in gold_df.columns if c not in non_source_cols]

# Qualite et stats globales
print("Nombre de jours:", len(gold_df))
print("Nombre de sources:", len(source_cols))
print("Total articles (somme total):", int(gold_df["total"].sum()))
print("Volume journalier - stats:")
display(gold_df["total"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).to_frame("total"))

# Top sources sur toute la periode
source_totals = gold_df[source_cols].sum().sort_values(ascending=False)
print("\nTop 15 sources (volume cumule)")
display(source_totals.head(15).to_frame("articles"))

# Concentration des sources (part cumulee)
source_share = (source_totals / source_totals.sum()).rename("share")
source_share_cum = source_share.cumsum().rename("cum_share")
concentration_df = pd.concat([source_share, source_share_cum], axis=1)
print("\nConcentration des sources (top 15)")
display(concentration_df.head(15).round(4))

# Jours atypiques via z-score simple
mu = gold_df["total"].mean()
sigma = gold_df["total"].std(ddof=0)
gold_df["z_total"] = (gold_df["total"] - mu) / (sigma if sigma != 0 else 1)
outliers = gold_df[gold_df["z_total"].abs() >= 3].sort_values("z_total", ascending=False)
print("\nJours atypiques (|z| >= 3)")
display(outliers[["date", "total", "z_total"]].head(20))

Nombre de jours: 4035
Nombre de sources: 16
Total articles (somme total): 4018310
Volume journalier - stats:


,total
count,4035.000000
mean,995.863693
std,802.582334
min,15.000000
10%,218.000000
25%,400.000000
50%,845.000000
75%,1349.500000
90%,1960.000000
99%,4083.420000



Top 15 sources (volume cumule)


,articles
Other,3887943
Cointelegraph,33863
CoinDesk,27073
Reuters,23228
CoinSpeaker,21037
Bitcoin.com,6619
Bloomberg,5830
BBC,3163
New York Times,2466
CNN,2389



Concentration des sources (top 15)


,share,cum_share
Other,0.9676,0.9676
Cointelegraph,0.0084,0.9760
CoinDesk,0.0067,0.9827
Reuters,0.0058,0.9885
CoinSpeaker,0.0052,0.9937
Bitcoin.com,0.0016,0.9954
Bloomberg,0.0015,0.9968
BBC,0.0008,0.9976
New York Times,0.0006,0.9982
CNN,0.0006,0.9988



Jours atypiques (|z| >= 3)


,date,total,z_total
1026,2017-12-11,7320,7.880712
1022,2017-12-07,7252,7.795975
1037,2017-12-22,6875,7.326183
1063,2018-01-17,6626,7.015896
1023,2017-12-08,6484,6.838945
1028,2017-12-13,6297,6.605918
1083,2018-02-06,6131,6.399060
1057,2018-01-11,6086,6.342984
1014,2017-11-29,5784,5.966652
1082,2018-02-05,5523,5.641412


In [5]:
# Tendances temporelles
plot_df = gold_df.sort_values("date").copy()
plot_df["ma7"] = plot_df["total"].rolling(7, min_periods=1).mean()
plot_df["ma30"] = plot_df["total"].rolling(30, min_periods=1).mean()

# Serie globale: total + moyennes mobiles
time_series_df = plot_df[["date", "total", "ma7", "ma30"]].melt(
    id_vars="date",
    var_name="serie",
    value_name="articles",
)

fig_total = px.line(
    time_series_df,
    x="date",
    y="articles",
    color="serie",
    title="Gold - Evolution du volume media",
)
fig_total.update_layout(legend_title_text="Serie")
fig_total.show()

# Top sources dans le temps
top_sources = source_totals.head(6).index.tolist()
top_sources_df = plot_df[["date"] + top_sources].melt(
    id_vars="date",
    var_name="source",
    value_name="articles",
)

fig_sources = px.line(
    top_sources_df,
    x="date",
    y="articles",
    color="source",
    title="Gold - Top 6 sources (series journalieres)",
)
fig_sources.update_layout(legend_title_text="Source")
fig_sources.show()

In [6]:
# Lecture business rapide (resume chiffrable)
coverage = pd.DataFrame({
    "metric": [
        "nb_days",
        "nb_sources",
        "total_articles",
        "median_daily_articles",
        "p90_daily_articles",
        "max_daily_articles",
        "top1_source_share",
        "top3_sources_share",
    ],
    "value": [
        int(len(gold_df)),
        int(len(source_cols)),
        int(gold_df["total"].sum()),
        float(gold_df["total"].median()),
        float(gold_df["total"].quantile(0.9)),
        float(gold_df["total"].max()),
        float(source_share.iloc[0]) if len(source_share) > 0 else np.nan,
        float(source_share.head(3).sum()) if len(source_share) > 0 else np.nan,
    ],
})

coverage

,metric,value
0,nb_days,4.035000e+03
1,nb_sources,1.600000e+01
2,total_articles,4.018310e+06
3,median_daily_articles,8.450000e+02
4,p90_daily_articles,1.960000e+03
5,max_daily_articles,7.320000e+03
6,top1_source_share,9.675568e-01
7,top3_sources_share,9.827213e-01


## Comparatif sources + ponderation

Regles de ponderation pour le ML:
- `1.0` pour les sources de reference (matching fuzzy contains):
  Reuters, Bloomberg, BBC, NYT, FT, WSJ, CNN, France 24, Le Monde, Les Echos, The Guardian
- `0.5` pour les sources qui matchent la regex crypto:
  `btc|bitcoin|coin|crypto|crypt|blockchain|defi|web3|token|altcoin`
- `0.1` pour toutes les autres sources, y compris `Other` / `Autres`

In [1]:
# Comparatif sources + ponderation personnalisee
import re
import unicodedata

# 1) Liste des sources depuis gold
all_gold_sources = [c for c in gold_df.columns if c not in {"date", "total", "z_total"}]

# 2) Sources de reference (poids 1.0) en fuzzy contains
trusted_terms = [
    "reuters",
    "bloomberg",
    "bbc",
    "new york times",
    "nyt",
    "financial times",
    "ft",
    "wall street journal",
    "wsj",
    "cnn",
    "france 24",
    "le monde",
    "les echos",
    "the guardian",
]

# 3) Regex crypto (poids 0.5)
crypto_pattern = re.compile(
    r"(btc|bitcoin|coin|crypto|crypt|blockchain|defi|web3|token|altcoin)",
    flags=re.IGNORECASE,
 )


def normalize_source_name(source_name: str) -> str:
    text = "" if source_name is None else str(source_name)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    return text.strip().lower()


def classify_source(source_name: str) -> str:
    s = normalize_source_name(source_name)
    if any(term in s for term in trusted_terms):
        return "trusted_mainstream"
    if crypto_pattern.search(s):
        return "crypto_regex"
    return "other_low"


def source_weight(source_class: str) -> float:
    weights = {
        "trusted_mainstream": 1.00,
        "crypto_regex": 0.50,
        "other_low": 0.10,
    }
    return weights[source_class]


source_comparison = pd.DataFrame({"source": all_gold_sources})
source_comparison["source_class"] = source_comparison["source"].apply(classify_source)
source_comparison["weight"] = source_comparison["source_class"].apply(source_weight)
source_comparison["raw_articles"] = source_comparison["source"].map(gold_df[all_gold_sources].sum())
source_comparison["weighted_articles"] = source_comparison["raw_articles"] * source_comparison["weight"]
source_comparison = source_comparison.sort_values("raw_articles", ascending=False).reset_index(drop=True)

print("Comparatif sources (classe + poids)")
display(source_comparison)

print("\nRepartition par classe de source")
display(source_comparison.groupby("source_class", dropna=False)["source"].count().to_frame("nb_sources"))

totals_check = pd.DataFrame(
    {
        "metric": ["raw_total", "weighted_total", "weighted_over_raw"],
        "value": [
            float(source_comparison["raw_articles"].sum()),
            float(source_comparison["weighted_articles"].sum()),
            float(
                source_comparison["weighted_articles"].sum()
                / source_comparison["raw_articles"].sum()
            ) if source_comparison["raw_articles"].sum() else np.nan,
        ],
    }
)
print("\nControle totals")
display(totals_check)

# Export CSV comparatif
source_comparison_path = DATA_DIR / "gold" / "gold_source_comparison_weighted.csv"
source_comparison.to_csv(source_comparison_path, index=False)
print("Export OK:", source_comparison_path)

NameError: name 'gold_df' is not defined

In [8]:
# Lissage 3 jours pour le ML (silver agregé + gold)
ML_DIR = DATA_DIR / "ml"
ML_DIR.mkdir(parents=True, exist_ok=True)

# A) Silver -> aggregation journaliere par domaine puis lissage 3 jours
silver_daily_domain = (
    silver_df.assign(date=pd.to_datetime(silver_df["date"], errors="coerce"))
    .dropna(subset=["date"])
    .groupby(["date", "domain"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
)

silver_pivot = (
    silver_daily_domain
    .pivot(index="date", columns="domain", values="count")
    .fillna(0)
    .sort_index()
)

silver_pivot_3d = silver_pivot.rolling(window=3, min_periods=1).mean()
silver_pivot_3d = silver_pivot_3d.reset_index()

silver_ml_path = ML_DIR / "silver_daily_domain_smoothed_3d.csv"
silver_pivot_3d.to_csv(silver_ml_path, index=False)

# B) Gold -> lissage 3 jours de toutes les colonnes sources + total
gold_ml = gold_df.copy().sort_values("date")
gold_feature_cols = [c for c in gold_ml.columns if c not in {"date", "z_total"}]
gold_ml_3d = gold_ml.copy()
gold_ml_3d[gold_feature_cols] = gold_ml_3d[gold_feature_cols].rolling(window=3, min_periods=1).mean()

gold_ml_path = ML_DIR / "gold_smoothed_3d.csv"
gold_ml_3d.to_csv(gold_ml_path, index=False)

print("Exports ML (lissage 3 jours) OK:")
print("-", silver_ml_path)
print("-", gold_ml_path)

display(gold_ml_3d.head(5))

Exports ML (lissage 3 jours) OK:
- /home/herrahj/test_crypto/crypto-prediction/data/ml/silver_daily_domain_smoothed_3d.csv
- /home/herrahj/test_crypto/crypto-prediction/data/ml/gold_smoothed_3d.csv


,date,BBC,Bitcoin.com,Bloomberg,CNN,CoinDesk,CoinSpeaker,Cointelegraph,Decrypt,Financial Times,France 24,Le Figaro,Le Monde,Les Échos,New York Times,Other,Reuters,total,z_total
0,2015-02-18,0.000000,0.0,0.000000,1.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,38.000000,0.000000,39.000000,-1.192379
1,2015-02-19,0.000000,0.0,0.500000,1.000000,3.500000,0.0,0.500000,0.0,0.0,0.0,0.0,0.0,0.0,1.000000,118.000000,0.500000,125.000000,-0.978044
2,2015-02-20,0.000000,0.0,0.333333,1.000000,4.666667,0.0,0.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.666667,131.000000,0.333333,138.333333,-1.035366
3,2015-02-21,0.000000,0.0,0.333333,0.666667,6.000000,0.0,1.666667,0.0,0.0,0.0,0.0,0.0,0.0,0.666667,163.000000,0.333333,172.666667,-1.064027
4,2015-02-22,0.333333,0.0,0.000000,0.333333,4.000000,0.0,1.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,140.333333,0.000000,146.333333,-1.076489


In [9]:
# (Optionnel) version gold ponderee puis lissee sur 3 jours
weighted_gold = gold_df.copy().sort_values("date")
weighted_sources = [s for s in source_comparison["source"] if s in weighted_gold.columns]

for s in weighted_sources:
    w = float(source_comparison.loc[source_comparison["source"] == s, "weight"].iloc[0])
    weighted_gold[s] = weighted_gold[s] * w

weighted_gold["total_weighted"] = weighted_gold[weighted_sources].sum(axis=1)
weighted_gold_3d = weighted_gold.copy()
weighted_cols = weighted_sources + ["total_weighted"]
weighted_gold_3d[weighted_cols] = weighted_gold_3d[weighted_cols].rolling(window=3, min_periods=1).mean()

weighted_gold_path = ML_DIR / "gold_weighted_smoothed_3d.csv"
weighted_gold_3d.to_csv(weighted_gold_path, index=False)
print("Export OK:", weighted_gold_path)

display(weighted_gold_3d[["date", "total_weighted"] + weighted_sources[:5]].head(5))

Export OK: /home/herrahj/test_crypto/crypto-prediction/data/ml/gold_weighted_smoothed_3d.csv


,date,total_weighted,Other,Cointelegraph,CoinDesk,Reuters,CoinSpeaker
0,2015-02-18,4.800000,3.800000,0.000000,0.000000,0.000000,0.0
1,2015-02-19,16.300000,11.800000,0.250000,1.750000,0.500000,0.0
2,2015-02-20,17.600000,13.100000,0.166667,2.333333,0.333333,0.0
3,2015-02-21,21.800000,16.300000,0.833333,3.000000,0.333333,0.0
4,2015-02-22,17.366667,14.033333,0.666667,2.000000,0.000000,0.0


In [ ]:
# Visualisations Plotly des CSV exportes (gold uniquement)

comparison_csv = pd.read_csv(source_comparison_path)
gold_smoothed_csv = pd.read_csv(gold_ml_path)
gold_weighted_csv = pd.read_csv(weighted_gold_path)

gold_smoothed_csv["date"] = pd.to_datetime(gold_smoothed_csv["date"], errors="coerce")
gold_weighted_csv["date"] = pd.to_datetime(gold_weighted_csv["date"], errors="coerce")

# 1) gold_source_comparison_weighted.csv -> bar chart raw vs weighted (top 20)
top_n = 20
comparison_top = comparison_csv.sort_values("raw_articles", ascending=False).head(top_n)
comparison_plot = comparison_top.melt(
    id_vars=["source", "source_class"],
    value_vars=["raw_articles", "weighted_articles"],
    var_name="metric",
    value_name="articles",
)
fig_comparison = px.bar(
    comparison_plot,
    x="source",
    y="articles",
    color="metric",
    barmode="group",
    hover_data=["source_class"],
    title="Gold Sources - Raw vs Weighted (Top 20)",
)
fig_comparison.update_layout(xaxis_title="Source", yaxis_title="Articles")
fig_comparison.show()

# 2) gold_smoothed_3d.csv -> line chart total lisse 3 jours
fig_gold_smoothed = px.line(
    gold_smoothed_csv.sort_values("date"),
    x="date",
    y="total",
    title="Gold Smoothed 3D - Total",
)
fig_gold_smoothed.update_layout(xaxis_title="Date", yaxis_title="Articles")
fig_gold_smoothed.show()

# 3) gold_weighted_smoothed_3d.csv -> line chart total_weighted lisse 3 jours
fig_gold_weighted = px.line(
    gold_weighted_csv.sort_values("date"),
    x="date",
    y="total_weighted",
    title="Gold Weighted Smoothed 3D - Total Weighted",
)
fig_gold_weighted.update_layout(xaxis_title="Date", yaxis_title="Weighted articles")
fig_gold_weighted.show()